In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

# Load your CSV
file_path = "/content/drive/MyDrive/sentiment_analysis_sample (2).csv"
df = pd.read_csv(file_path)

# View columns
df.head()

,person1_id,person2_id,total_messages,conversation_start,last_message,person1_is_mentor,person1_is_entrepreneur,person1_country,person1_gender,person1_ethnicity,...,total_engagement_words,engagement_density,positive_mentorship_words,negative_mentorship_words,mentorship_sentiment_ratio,first_half_sentiment,second_half_sentiment,sentiment_progression,sender_balance,detected_language_y
0,1649650,1669943,2,2025-03-17 23:27:31.241 +0530,2025-07-13 18:02:27.127 +0530,True,False,EG,m,NaN,...,2,1.000,0,0,0.0,0.000000,0.693690,0.693690,0.50,ru
1,1056243,1915097,4,2025-07-07 18:08:27.900 +0530,2025-07-07 19:35:53.537 +0530,True,False,KE,m,NaN,...,3,0.750,8,0,8.0,0.803521,0.563917,-0.239604,0.25,en
2,1559384,1913488,8,2025-07-05 21:32:15.380 +0530,2025-07-07 18:47:26.810 +0530,False,True,US,f,Latino or Hispanic,...,9,1.125,0,1,0.0,0.000000,0.137024,0.137024,0.25,en
3,1626417,1910835,6,2025-07-01 11:35:56.148 +0530,2025-07-09 17:43:14.057 +0530,True,False,KE,m,NaN,...,24,4.000,7,0,7.0,0.484308,0.000000,-0.484308,0.50,en
4,1136103,1629049,4,2025-06-21 03:47:16.770 +0530,2025-07-16 16:45:46.154 +0530,True,False,SA,m,NaN,...,0,0.000,0,0,0.0,0.738764,0.455549,-0.283215,0.25,ar


In [3]:
!pip install transformers torch pandas tqdm requests

In [4]:
!pip install --force-reinstall transformers==4.42.4
import importlib
import transformers
importlib.reload(transformers)
from transformers import pipeline


  Using cached transformers-4.42.4-py3-none-any.whl.metadata (43 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached huggingface_hub-0.35.3-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2025.9.18-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-2

KeyboardInterrupt: 

In [1]:
from transformers import pipeline

# Load RoBERTa-large-MNLI model
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",  # or "roberta-large-mnli"
    device_map="auto"
)

labels = ["internal motivation", "external motivation"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
from tqdm import tqdm

def classify_motivation(text):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return None, 0, 0

    result = classifier(text, labels)
    scores = dict(zip(result["labels"], result["scores"]))
    internal = scores.get("internal motivation", 0)
    external = scores.get("external motivation", 0)

    # Threshold-based multi-label logic
    threshold = 0.5
    if internal >= threshold and external >= threshold:
        category = "both"
    elif internal >= threshold:
        category = "internal"
    elif external >= threshold:
        category = "external"
    else:
        category = "uncertain"

    return category, internal, external

In [6]:
tqdm.pandas()

for person in ["person1", "person2"]:
    col = f"{person}_motivation_mentorship"
    df[[f"{person}_category", f"{person}_internal_score", f"{person}_external_score"]] = (
        df[col].progress_apply(lambda x: pd.Series(classify_motivation(x)))
    )

100%|██████████| 10/10 [00:18<00:00,  1.84s/it]


In [7]:
import numpy as np

thresholds = np.linspace(0.3, 0.7, 5)
for t in thresholds:
    internal = (df["person1_internal_score"] >= t).sum()
    external = (df["person1_external_score"] >= t).sum()
    print(f"Threshold {t:.2f} → Internal: {internal}, External: {external}")


Threshold 0.30 → Internal: 1, External: 3
Threshold 0.40 → Internal: 1, External: 3
Threshold 0.50 → Internal: 1, External: 2
Threshold 0.60 → Internal: 0, External: 2
Threshold 0.70 → Internal: 0, External: 2


In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# --- Check if true label columns exist ---
if "person1_true_label" in df.columns:
    print("✅ Found ground truth labels. Calculating real accuracy metrics...\n")
    y_true = df["person1_true_label"]
    y_pred = df["person1_category"]

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-score: {f1:.2f}")

else:
    print("⚠️ No ground truth labels found — switching to confidence-based evaluation.\n")

    # --- Average confidence ---
    avg_internal = df["person1_internal_score"].mean()
    avg_external = df["person1_external_score"].mean()
    print(f"Average Internal Confidence: {avg_internal:.2f}")
    print(f"Average External Confidence: {avg_external:.2f}")

    # --- Dominance (which motivation is more common) ---
    dominance = np.mean(df["person1_internal_score"] > df["person1_external_score"])
    print(f"% of responses classified as Internally motivated: {dominance*100:.1f}%")

    # --- Mentor–Mentee category agreement ---
    if "person2_category" in df.columns:
        agreement = (df["person1_category"] == df["person2_category"]).mean()
        print(f"Mentor–Mentee Motivation Agreement: {agreement*100:.1f}%")

    # --- Confidence gap as a pseudo-quality metric ---
    df["confidence_gap"] = abs(df["person1_internal_score"] - df["person1_external_score"])
    quality_score = df["confidence_gap"].mean()
    print(f"Overall Confidence Gap (higher = clearer classification): {quality_score:.2f}")

⚠️ No ground truth labels found — switching to confidence-based evaluation.

Average Internal Confidence: 0.08
Average External Confidence: 0.22
% of responses classified as Internally motivated: 10.0%
Mentor–Mentee Motivation Agreement: 0.0%
Overall Confidence Gap (higher = clearer classification): 0.16


In [11]:
dominance = np.mean(df["person1_internal_score"] > df["person1_external_score"])
print(f"% of responses classified as internally motivated: {dominance*100:.1f}%")

% of responses classified as internally motivated: 10.0%


In [12]:
agreement = (df["person1_category"] == df["person2_category"]).mean()
print(f"Mentor–Mentee Motivation Agreement: {agreement*100:.1f}%")

Mentor–Mentee Motivation Agreement: 0.0%


In [13]:
df["confidence_gap"] = abs(df["person1_internal_score"] - df["person1_external_score"])
quality_score = df["confidence_gap"].mean()
print(f"Overall Confidence Gap (higher = clearer classification): {quality_score:.2f}")


Overall Confidence Gap (higher = clearer classification): 0.16


In [14]:
output_path = "/content/motivation_classified.csv"
df.to_csv(output_path, index=False)
print(f"✅ Results saved to {output_path}")

from google.colab import files
files.download(output_path)

✅ Results saved to /content/motivation_classified.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>